# Fruit Classification — TinyML Pipeline
RGB-based fruit classification using traditional ML, designed for microcontroller deployment.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import os

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.manifold import TSNE
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("xgboost not installed — XGBoost section will be skipped.")

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

## 2. Load Data

In [ ]:
# -------------------------------------------------------------------
# Option A — single CSV with a label column
# df = pd.read_csv('fruits.csv')
#
# Option B — one CSV per class inside a folder
# -------------------------------------------------------------------

FRUITS_DIR = 'fruits'   # folder containing apple.csv, banana.csv, etc.

frames = []
for fname in os.listdir(FRUITS_DIR):
    if not fname.endswith('.csv'):
        continue
    label = os.path.splitext(fname)[0]
    tmp = pd.read_csv(os.path.join(FRUITS_DIR, fname))
    tmp['fruit'] = label
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
df.columns = df.columns.str.strip().str.lower()

print(df.shape)
df.head(10)

## 3. EDA

### 3.1 Basic statistics

In [ ]:
df.describe().round(2)

In [ ]:
df['fruit'].value_counts()

In [ ]:
print("Nulls per column:")
print(df.isnull().sum())
print(f"\nDuplicated rows: {df.duplicated().sum()}")

### 3.2 Class balance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
counts = df['fruit'].value_counts()
bars = ax.bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)), edgecolor='white')
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_xlabel('Fruit')
ax.set_ylabel('Samples')
ax.set_title('Class Distribution')
ax.set_ylim(0, counts.max() * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

### 3.3 Feature distributions by class

In [ ]:
features = ['r', 'g', 'b']
palette = sns.color_palette('muted', df['fruit'].nunique())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    for i, (label, grp) in enumerate(df.groupby('fruit')):
        ax.hist(grp[feat], bins=18, alpha=0.65, label=label, color=palette[i], edgecolor='white')
    ax.set_title(f'Channel: {feat.upper()}')
    ax.set_xlabel('Intensity')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
sns.despine()
fig.suptitle('RGB Intensity Distributions per Fruit', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Box plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    sns.boxplot(data=df, x='fruit', y=feat, palette='muted', ax=ax,
                flierprops=dict(marker='o', markersize=4, alpha=0.5))
    ax.set_title(f'Channel: {feat.upper()}')
    ax.set_xlabel('')
sns.despine()
fig.suptitle('Per-channel Spread by Class', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Pair plot

In [ ]:
g = sns.pairplot(df, hue='fruit', vars=features, palette='muted',
                 plot_kws=dict(alpha=0.6, s=25, edgecolor='none'),
                 diag_kind='kde')
g.fig.suptitle('Pair Plot — RGB Features', y=1.02, fontweight='bold')
plt.show()

### 3.6 Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation')
plt.tight_layout()
plt.show()

### 3.7 t-SNE scatter

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
embed = tsne.fit_transform(df[features].values)

fig, ax = plt.subplots(figsize=(7, 5))
for i, (label, grp_idx) in enumerate(df.groupby('fruit').groups.items()):
    pts = embed[grp_idx]
    ax.scatter(pts[:, 0], pts[:, 1], label=label, alpha=0.75,
               color=palette[i], edgecolors='none', s=35)
ax.set_title('t-SNE Projection (2D)')
ax.set_xlabel('Component 1')
ax.set_ylabel('Component 2')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 4. Data Cleaning

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Shape after cleaning: {df.shape}")

In [ ]:
# IQR-based outlier check per feature — flag rows 3σ outside
def flag_outliers(frame, cols, threshold=3.0):
    mask = pd.Series(False, index=frame.index)
    for col in cols:
        z = (frame[col] - frame[col].mean()) / frame[col].std()
        mask |= z.abs() > threshold
    return mask

outlier_mask = flag_outliers(df, features)
print(f"Outlier rows detected: {outlier_mask.sum()}")
df = df[~outlier_mask].reset_index(drop=True)
print(f"Shape after outlier removal: {df.shape}")

## 5. Feature Engineering

### 5.1 Encode targets & split

In [ ]:
X = df[features].values
y_raw = df['fruit'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = list(le.classes_)
print("Classes:", class_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

### 5.2 Min-max normalization

In [ ]:
scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Scaled train range — min:", X_train_sc.min().round(4), " max:", X_train_sc.max().round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (X_plot, title) in zip(axes, [(X_train, 'Raw'), (X_train_sc, 'Min-Max Scaled')]):
    for i, feat in enumerate(features):
        ax.hist(X_plot[:, i], bins=20, alpha=0.6, label=feat.upper(), edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

### 5.3 Feature importance — univariate F-score

In [ ]:
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_train_sc, y_train)

fi_df = pd.DataFrame({'feature': features, 'f_score': selector.scores_, 'p_value': selector.pvalues_})
fi_df = fi_df.sort_values('f_score', ascending=False)
print(fi_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(fi_df['feature'], fi_df['f_score'], color=sns.color_palette('muted', 3))
ax.set_xlabel('F-score')
ax.set_title('Univariate Feature Importance')
sns.despine()
plt.tight_layout()
plt.show()

### 5.4 Sequential feature selection

In [ ]:
sfs = SequentialFeatureSelector(
    RandomForestClassifier(n_estimators=50, random_state=SEED),
    direction='forward',
    scoring='accuracy',
    cv=5,
    n_features_to_select='auto',
    tol=0.0
)
sfs.fit(X_train_sc, y_train)
selected_features = [features[i] for i in range(len(features)) if sfs.get_support()[i]]
print("Selected features:", selected_features)

### 5.5 Recursive feature elimination

In [ ]:
rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=50, random_state=SEED),
    n_features_to_select=2
)
rfe.fit(X_train_sc, y_train)
rfe_selected = [features[i] for i in range(len(features)) if rfe.support_[i]]
print("RFE selected:", rfe_selected)
print("Feature ranking:", dict(zip(features, rfe.ranking_)))

All three channels are used in the final pipeline — the dataset is small and there is no dimensionality concern.

## 6. Classification Models

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate(name, estimator, X_tr, y_tr, X_te, y_te):
    cv_scores = cross_val_score(estimator, X_tr, y_tr, cv=cv, scoring='f1_macro')
    estimator.fit(X_tr, y_tr)
    y_pred = estimator.predict(X_te)
    test_acc = accuracy_score(y_te, y_pred)
    test_f1  = f1_score(y_te, y_pred, average='macro')
    print(f"{'─'*60}")
    print(f"  {name}")
    print(f"  CV F1 (macro): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Test Acc:      {test_acc:.4f}    Test F1: {test_f1:.4f}")
    print()
    print(classification_report(y_te, y_pred, target_names=class_names))
    return estimator, y_pred, cv_scores, test_acc, test_f1

### 6.1 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=SEED)
dt_model, dt_pred, dt_cv, dt_acc, dt_f1 = evaluate(
    'Decision Tree', dt, X_train_sc, y_train, X_test_sc, y_test
)

In [ ]:
print(export_text(dt_model, feature_names=features))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt_model, feature_names=features, class_names=class_names,
          filled=True, rounded=True, ax=ax, fontsize=8, impurity=False)
ax.set_title('Decision Tree Structure', fontsize=13)
plt.tight_layout()
plt.show()

### 6.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=2,
                             random_state=SEED, n_jobs=-1)
rf_model, rf_pred, rf_cv, rf_acc, rf_f1 = evaluate(
    'Random Forest', rf, X_train_sc, y_train, X_test_sc, y_test
)

In [ ]:
importances = rf_model.feature_importances_
perm = permutation_importance(rf_model, X_test_sc, y_test, n_repeats=30, random_state=SEED)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].barh(features, importances, color=sns.color_palette('muted', 3))
axes[0].set_title('Gini Importance')
axes[0].set_xlabel('Mean Decrease in Impurity')

axes[1].barh(features, perm.importances_mean,
             xerr=perm.importances_std, color=sns.color_palette('muted', 3))
axes[1].set_title('Permutation Importance (test set)')
axes[1].set_xlabel('Mean Accuracy Decrease')

sns.despine()
plt.tight_layout()
plt.show()

### 6.3 XGBoost

In [ ]:
if XGBOOST_AVAILABLE:
    xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                        use_label_encoder=False, eval_metric='mlogloss',
                        random_state=SEED, verbosity=0)
    xgb_model, xgb_pred, xgb_cv, xgb_acc, xgb_f1 = evaluate(
        'XGBoost', xgb, X_train_sc, y_train, X_test_sc, y_test
    )
else:
    xgb_acc = xgb_f1 = None
    print("Skipped — xgboost not installed.")

### 6.4 Logistic Regression

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=500, multi_class='multinomial',
                        solver='lbfgs', random_state=SEED)
lr_model, lr_pred, lr_cv, lr_acc, lr_f1 = evaluate(
    'Logistic Regression', lr, X_train_sc, y_train, X_test_sc, y_test
)

coef_df = pd.DataFrame(lr_model.coef_, columns=features, index=class_names)
print("\nCoefficients:")
print(coef_df.round(4))

### 6.5 SVM

In [ ]:
svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED)
svm_model, svm_pred, svm_cv, svm_acc, svm_f1 = evaluate(
    'SVM (RBF)', svm, X_train_sc, y_train, X_test_sc, y_test
)

## 7. Model Comparison

In [ ]:
results = {
    'Decision Tree':     (dt_acc,  dt_f1),
    'Random Forest':     (rf_acc,  rf_f1),
    'Logistic Reg.':     (lr_acc,  lr_f1),
    'SVM (RBF)':         (svm_acc, svm_f1),
}
if XGBOOST_AVAILABLE and xgb_acc is not None:
    results['XGBoost'] = (xgb_acc, xgb_f1)

cmp_df = pd.DataFrame(results, index=['Accuracy', 'F1 Macro']).T
cmp_df = cmp_df.sort_values('F1 Macro', ascending=False)
print(cmp_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = sns.color_palette('muted', len(cmp_df))

for ax, metric in zip(axes, ['Accuracy', 'F1 Macro']):
    bars = ax.bar(cmp_df.index, cmp_df[metric], color=colors, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.set_ylim(cmp_df[metric].min() * 0.95, 1.02)
    ax.set_title(metric)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=20)

sns.despine()
fig.suptitle('Test Set Performance', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Confusion Matrices

In [ ]:
model_preds = {
    'Decision Tree':  dt_pred,
    'Random Forest':  rf_pred,
    'Logistic Reg.':  lr_pred,
    'SVM (RBF)':      svm_pred,
}
if XGBOOST_AVAILABLE and xgb_acc is not None:
    model_preds['XGBoost'] = xgb_pred

n_models = len(model_preds)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, preds) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Confusion Matrices — Test Set', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Final Pipeline (FruitChain)

Random Forest is chosen as the final estimator. The pipeline is retrained on the full dataset.

In [ ]:
fruit_chain = Pipeline([
    ('scaler', MinMaxScaler()),
    ('clf',    RandomForestClassifier(n_estimators=100, max_depth=8,
                                       min_samples_leaf=2, random_state=SEED, n_jobs=-1))
])

fruit_chain.fit(X, y)

# sanity check on full dataset
y_full_pred = fruit_chain.predict(X)
print(f"Train accuracy (full dataset): {accuracy_score(y, y_full_pred):.4f}")
print(f"Train F1 macro  (full dataset): {f1_score(y, y_full_pred, average='macro'):.4f}")

In [ ]:
# Verify the pipeline on a held-out split
cv_final = cross_val_score(fruit_chain, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                            scoring='f1_macro')
print(f"5-fold CV F1 macro: {cv_final.mean():.4f} ± {cv_final.std():.4f}")

## 10. Export — FruitChain.h

In [ ]:
fitted_scaler = fruit_chain.named_steps['scaler']
fitted_rf     = fruit_chain.named_steps['clf']

scale_min = fitted_scaler.data_min_.tolist()
scale_range = (fitted_scaler.data_max_ - fitted_scaler.data_min_).tolist()

n_classes  = len(class_names)
n_features = len(features)

def tree_to_c(tree, node=0, depth=0, feature_names=None):
    indent = '  ' * (depth + 2)
    left  = tree.children_left[node]
    right = tree.children_right[node]
    feat  = tree.feature[node]
    thr   = tree.threshold[node]

    if left == right:  # leaf
        values = tree.value[node][0]
        pred   = int(np.argmax(values))
        return f"{indent}return {pred};\n"

    fname = feature_names[feat] if feature_names else f"x[{feat}]"
    code  = f"{indent}if (x[{feat}] <= {thr:.6f}f) {{\n"
    code += tree_to_c(tree, left,  depth + 1, feature_names)
    code += f"{indent}}} else {{\n"
    code += tree_to_c(tree, right, depth + 1, feature_names)
    code += f"{indent}}}\n"
    return code


tree_functions = []
for i, estimator in enumerate(fitted_rf.estimators_):
    body = tree_to_c(estimator.tree_, feature_names=features)
    tree_functions.append(
        f"  static int tree_{i}(const float* x) {{\n{body}  return 0;\n  }}"
    )

header_lines = []
header_lines.append("// Auto-generated by fruit_classification.ipynb")
header_lines.append("// FruitChain — MinMaxScaler + Random Forest")
header_lines.append("#pragma once")
header_lines.append("#include <stdint.h>")
header_lines.append("")
header_lines.append("namespace tinyml4all {")
header_lines.append("")
header_lines.append("class FruitChain {")
header_lines.append("public:")

# label strings
labels_str = ', '.join(f'"{c}"' for c in class_names)
header_lines.append(f"  static constexpr int   NUM_CLASSES  = {n_classes};")
header_lines.append(f"  static constexpr int   NUM_FEATURES = {n_features};")
header_lines.append(f"  static constexpr int   NUM_TREES    = {len(fitted_rf.estimators_)};")
header_lines.append(f"  const char* CLASS_NAMES[{n_classes}] = {{ {labels_str} }};")
header_lines.append("")
header_lines.append("  struct Output { const char* label; int idx; } output;"
                    "")
header_lines.append("")

# scale parameters
min_vals  = ', '.join(f'{v:.6f}f' for v in scale_min)
rng_vals  = ', '.join(f'{v:.6f}f' for v in scale_range)
header_lines.append(f"  const float SCALE_MIN[{n_features}]   = {{ {min_vals} }};")
header_lines.append(f"  const float SCALE_RANGE[{n_features}] = {{ {rng_vals} }};")
header_lines.append("")

# operator()
header_lines.append("  bool operator()(float r, float g, float b) {")
header_lines.append(f"    float raw[{n_features}] = {{r, g, b}};")
header_lines.append(f"    float x[{n_features}];")
header_lines.append(f"    for (int i = 0; i < {n_features}; i++)")
header_lines.append("      x[i] = (raw[i] - SCALE_MIN[i]) / SCALE_RANGE[i];")
header_lines.append("")
header_lines.append(f"    int votes[{n_classes}] = {{0}};")
header_lines.append(f"    for (int t = 0; t < {len(fitted_rf.estimators_)}; t++)")
header_lines.append("      votes[predict_tree(t, x)]++;")
header_lines.append("")
header_lines.append(f"    int best = 0;")
header_lines.append(f"    for (int c = 1; c < {n_classes}; c++)")
header_lines.append("      if (votes[c] > votes[best]) best = c;")
header_lines.append("    output.idx   = best;")
header_lines.append("    output.label = CLASS_NAMES[best];")
header_lines.append("    return true;")
header_lines.append("  }")
header_lines.append("")

# dispatch
dispatch = "  int predict_tree(int t, const float* x) {\n    switch(t) {\n"
for i in range(len(fitted_rf.estimators_)):
    dispatch += f"      case {i}: return tree_{i}(x);\n"
dispatch += "      default: return 0;\n    }\n  }"
header_lines.append(dispatch)
header_lines.append("")
header_lines.append("private:")
header_lines.extend(tree_functions)
header_lines.append("};")  # class
header_lines.append("")    # namespace
header_lines.append("} // namespace tinyml4all")

header_code = '\n'.join(header_lines)
print(header_code[:2000], "\n... (truncated)")

In [ ]:
with open('FruitChain.h', 'w') as f:
    f.write(header_code)

print(f"Saved FruitChain.h  ({len(header_code):,} chars)")

## 11. Quick Inference Test

In [ ]:
test_samples = [
    ([38, 23, 18], 'orange'),
    ([17, 12,  9], 'banana'),
    ([25, 30, 20], 'apple'),
]

for raw, expected in test_samples:
    x_sc    = fitted_scaler.transform([raw])
    pred_id = fitted_rf.predict(x_sc)[0]
    pred_lbl = class_names[pred_id]
    status  = '✓' if pred_lbl == expected else '✗'
    print(f"{status}  RGB{tuple(raw)}  →  {pred_lbl}  (expected: {expected})")

## 12. Arduino Sketch Reference

Place `FruitChain.h` inside the Arduino sketch folder and use the snippet below.

```cpp
#include <Arduino_APDS9960.h>
#include <tinyml4all.h>
#include "./FruitChain.h"

tinyml4all::APDS9960    sensor;
tinyml4all::FruitChain  chain;

void setup() {
    Serial.begin(115200);
    while (!Serial);
    sensor.begin();
}

void loop() {
    sensor.readColor();
    if (!chain(sensor.r, sensor.g, sensor.b)) return;
    Serial.print("Detected: ");
    Serial.println(chain.output.label);
    delay(1000);
}
```

> **Note:** The classifier only knows the classes it was trained on. Point the sensor at an unrecognised object and it will still return the closest match — collect a `none` class if a rejection option is needed.